### Data Download & Preparation: Cleaning & Filtering IT Resumes

We will first,   
- Download the Kaggle Resume dataset. 

Then,
-  filters for IT & Tech roles, parses raw HTML using BeautifulSoup, and exports a clean dataset for the matching engine.

In [1]:
# Import necessary libraries
import os
import re
from pathlib import Path
import pandas as pd
import kagglehub
import html

# setting for inspecting long text strings
pd.set_option("display.max_colwidth", 200)

In [2]:
# folder for raw data
raw_dir = Path("raw")
raw_dir.mkdir(parents=True, exist_ok=True)

# download the dataset from Kaggle 
print("Downloading snehaanbhawal/resume-dataset via kagglehub...")
print("=="*40)

dataset_dir = kagglehub.dataset_download("snehaanbhawal/resume-dataset")
print(f"Dataset downloaded to local cache: {dataset_dir}")

Dataset downloaded to local cache: /Users/sujansharma/.cache/kagglehub/datasets/snehaanbhawal/resume-dataset/versions/1


In [3]:
# dataset_dir = "raw"
dataset_path = Path(dataset_dir)
csv_files = list(dataset_path.rglob("*.csv"))

print("CSV files found in downloaded data directory:")
for f in csv_files:
    print(f" - {f.name}")


csv_file_path = csv_files[0]

CSV files found in downloaded data directory:
 - Resume.csv


In [4]:
df = pd.read_csv(csv_file_path)

print(f"Total rows loaded: {len(df)}")
df.head(3)

Total rows loaded: 2484


,ID,Resume_str,Resume_html,Category
0,16852973,HR ADMINISTRATOR/MARKETING ASSOCIATE\n\nHR ADMINISTRATOR Summary Dedicated Customer Service Manager with 15+ years of experience in Hospitality and Customer Service Management. ...,"<div class=""fontsize fontface vmargins hmargins linespacing pagesize"" id=""document""> <div class=""section firstsection"" id=""SECTION_NAME500375979"" style=""\n padding-top:0px;\n ""> <div class...",HR
1,22323967,"HR SPECIALIST, US HR OPERATIONS Summary Versatile media professional with background in Communications, Marketing, Human Resources and Technology. Experience 09/201...","<div class=""fontsize fontface vmargins hmargins linespacing pagesize"" id=""document""> <div class=""section firstsection"" id=""SECTION_NAME911808366"" style=""padding-top:0px;""> <div class=""paragraph PA...",HR
2,33176873,"HR DIRECTOR Summary Over 20 years experience in recruiting, 15 plus years in Human Resources Executive Management, 5 years of HRIS development and maintenance 4 years work...","<div class=""fontsize fontface vmargins hmargins linespacing pagesize"" id=""document""> <div class=""section firstsection"" id=""SECTION_NAME1008511259"" style=""padding-top:0px;""> <div class=""paragraph P...",HR


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2484 entries, 0 to 2483
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   ID           2484 non-null   int64
 1   Resume_str   2484 non-null   str  
 2   Resume_html  2484 non-null   str  
 3   Category     2484 non-null   str  
dtypes: int64(1), str(3)
memory usage: 77.8 KB


In [6]:
print("Top 15 resume categories:")
print(df["Category"].value_counts().head(15))

Top 15 resume categories:
Category
INFORMATION-TECHNOLOGY    120
BUSINESS-DEVELOPMENT      120
ADVOCATE                  118
CHEF                      118
FINANCE                   118
ENGINEERING               118
ACCOUNTANT                118
FITNESS                   117
AVIATION                  117
SALES                     116
HEALTHCARE                115
CONSULTANT                115
BANKING                   115
CONSTRUCTION              112
PUBLIC-RELATIONS          111
Name: count, dtype: int64


### Filtering IT & Tech Categories
Separating the resumes belonging to IT, Software Engineering, and Digital Media to build an IT-specific benchmark dataset.

In [7]:
it_categories = ["INFORMATION-TECHNOLOGY", "ENGINEERING", "DIGITAL-MEDIA"]

# Filter dataset
it_df = df[df["Category"].isin(it_categories)].copy().reset_index(drop=True)

print(f"Filtered down from {len(df)} to {len(it_df)} IT resumes.")
print("\nBreakdown by selected category:")
print(it_df["Category"].value_counts())

Filtered down from 2484 to 334 IT resumes.

Breakdown by selected category:
Category
INFORMATION-TECHNOLOGY    120
ENGINEERING               118
DIGITAL-MEDIA              96
Name: count, dtype: int64


In [ ]:
# checking one
it_df[it_df['Category']=="DIGITAL-MEDIA"]['Resume_html'].iloc[0]

'<div class="fontsize fontface vmargins hmargins linespacing pagesize" id="document"> <div class="section firstsection" id="SECTION_NAME921919160" style="padding-top:0px;"> <div class="paragraph PARAGRAPH_NAME firstparagraph" id="PARAGRAPH_921919160_1_605691096" style="padding-top:0px;"> <div class="name thinbottomborder" itemprop="name"> <span class="field" id="921919160FNAM1"> </span> <span> </span> <span class="field" id="921919160LNAM1"> DIGITAL MEDIA BUYER</span> </div> <div class="botBorder"> </div> </div> </div> <div class="section" id="SECTION_SUMM921919166" style="padding-top:0px;"> <div class="heading"> <div class="sectiontitle" id="SECTNAME_SUMM921919166"> Professional Summary</div> </div> <div class="paragraph firstparagraph" id="PARAGRAPH_921919166_1_605694071" style="padding-top:0px;"> <div class="field singlecolumn" id="921919166FRFM1"> <span class=""> Versatile digital marketer\xa0bringing </span> </div> </div> </div> <div class="section" id="SECTION_HILT921919164" styl

Since the resumes are in HTML format, it is easier to understand in Markdown format. 

### Now, Clearning HTML and converting to Markdown format

In [8]:
from markdownify import markdownify as md
import html

In [9]:
def clean_html_to_markdown(html_str: str, raw_text_fallback: str = "") -> str:
    """
        Cleaning inconsistent html, Converting it to Md and normalize.
    """
    if pd.isna(html_str) or not str(html_str).strip():
        text = str(raw_text_fallback) if pd.notna(raw_text_fallback) else ""
        return text.strip()

    # Unescape HTML entities (e.g., &amp; -> &, &nbsp; -> space)
    text = html.unescape(str(html_str))

    # Convert to Markdown (ATX headers: #, ##, ###)
    markdown_text = md(text, strip=["img", "a", "script", "style"], heading_style="ATX")

    # Clean up formatting 
    markdown_text = markdown_text.replace("\xa0", " ").replace("\r\n", "\n")
    markdown_text = re.sub(r"\n{3,}", "\n\n", markdown_text)
    
    # Trim leading/trailing spaces 
    lines = [line.strip() for line in markdown_text.split("\n")]
    
    return "\n".join(lines).strip()

In [10]:
# Convert HTML to Markdown across all resumes
print("Transforming HTML resumes into Markdown...")
it_df["resume_md"] = it_df.apply(
    lambda row: clean_html_to_markdown(row.get("Resume_html"), row.get("Resume_str")),
    axis=1
)

# final df
final_df = it_df[["ID", "Category", "resume_md"]]

print(f"Data preparation complete!")
print(f"Original data length: {len(df)} | Final cleaned data length: {len(final_df)}")

Transforming HTML resumes into Markdown...
Data preparation complete!
Original data length: 2484 | Final cleaned data length: 334


### Now, Save final IT related with markdown column resume dataset

In [12]:
output_dir = Path("processed")
output_dir.mkdir(exist_ok=True)

output_file = output_dir / "final_resumes.csv"
final_df.to_csv(output_file, index=False, encoding="utf-8")

print(f"Saved processed dataset to {output_file.resolve()}")

Saved processed dataset to /Users/sujansharma/Documents/0Study_Files/Python-Programming/ResuMatch/data/data_preparation/processed/final_resumes.csv
